# Weeks 1–4 Reflection Code

All of the code behind the Week 1–4 reflections, one section per week. The written answers are in `Weeks_1-4_Reflections.pdf`.

Run from the `Module-6-Assignments` folder so the relative `csvs/` paths resolve. Cells run top to bottom.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors

CSVS = Path("csvs")

## Week 1 — Matching (`homework_1.2.csv`)

### Q1. Is the farthest match distance too far to be a meaningful match? How can you decide this?

Match each treated row (X = 1) to its nearest control (X = 0) on Z, then look at how far the matches are and whether the far ones change the estimate.

In [ ]:
match = pd.read_csv(CSVS / "homework_1.2.csv")
treated = match[match["X"] == 1]
control = match[match["X"] == 0]
treated_y = treated["Y"].to_numpy()
control_y = control["Y"].to_numpy()

# Approach A: nearest control on Z, with replacement.
nn = NearestNeighbors(n_neighbors=1).fit(control[["Z"]])
distances, indices = nn.kneighbors(treated[["Z"]])
distances = distances.ravel()
matched_y = control_y[indices.ravel()]

print(f"farthest match distance:     {distances.max():.4f}")
print(f"median match distance:       {np.median(distances):.4f}")
print(f"farthest, in SDs of Z:       {distances.max() / match['Z'].std():.2f}  (rule of thumb: 0.2)")
print(f"treated above every control: {(treated['Z'] > control['Z'].max()).sum()} of {len(treated)}")

In [ ]:
# Drop matches farther than a caliper and see whether the effect moves.
print(f"{'caliper':<9} {'matches kept':>13} {'effect':>8}")
for caliper in [np.inf, 0.05, 0.02]:
    keep = distances <= caliper
    effect = (treated_y[keep] - matched_y[keep]).mean()
    print(f"{caliper:<9} {keep.sum():>13} {effect:8.3f}")

### Q2. Invent your own type of matching similar to (A) and (B)

Weighted matching: like (B), take every control within a radius of each treated row, but weight each one by `1 - distance / radius` so closer controls count more. Each treated row's match is the weighted average Y of its controls.

In [ ]:
def radius_matching(radius=0.2, weighted=True):
    """Each treated row vs. its controls within `radius`. weighted=False is approach B."""
    nn = NearestNeighbors(radius=radius).fit(control[["Z"]])
    dists, groups = nn.radius_neighbors(treated[["Z"]])
    effects = []
    for y, d, group in zip(treated_y, dists, groups):
        if len(group):
            weights = 1 - d / radius if weighted else None
            effects.append(y - np.average(control_y[group], weights=weights))
    return np.mean(effects)


print(f"(A) nearest neighbor:        {(treated_y - matched_y).mean():.3f}")
print(f"(B) radius, equal weights:   {radius_matching(weighted=False):.3f}")
print(f"(C) radius, closer weighted: {radius_matching(weighted=True):.3f}")

## Week 2 — Fixed effects and the bootstrap

### Q1. Invent an example situation that would use fixed effects.

Written answer only (coffee chain menu rollout), no code.

### Q2. Bootstrap the variance in the mean of a Pareto distribution. As the sample size grows, what happens to that variance?

Draw a sample of size n from a Pareto, resample it with replacement B times, and take the variance of those B means. Repeat as n grows to read off the rate.

Pareto(shape a, scale 1) has a mean only when a > 1 and a finite variance only when a > 2, so a = 3.0 converges at the usual 1/n rate and a = 1.5 does not.

In [ ]:
SHAPES = {"a = 3.0 (finite variance)": 3.0, "a = 1.5 (infinite variance)": 1.5}
SIZES = [100, 400, 1600, 6400, 25600]  # quadruples, so 1/n predicts a ratio of 0.25
N_BOOTSTRAP = 5_000
N_REPLICATES = 40  # one sample alone is far too noisy on a heavy tail
SEED = 0


def pareto_sample(rng, n, a):
    """numpy's pareto() is Lomax and starts at 0, so shift it to start at 1."""
    return 1.0 + rng.pareto(a, n)


def bootstrap_mean_variance(sample, rng):
    """Resample with replacement B times, return the variance of the B means."""
    idx = rng.integers(0, len(sample), size=(N_BOOTSTRAP, len(sample)))
    return sample[idx].mean(axis=1).var(ddof=1)


def theoretical(a, n):
    """var(X)/n, which only exists for a > 2."""
    return a / ((a - 1) ** 2 * (a - 2)) / n if a > 2 else float("nan")

In [ ]:
def run(label, a):
    rng = np.random.default_rng(SEED)
    print(f"\n{label}")
    print(f"{'n':>7} {'bootstrap var(mean)':>21} {'theory var(X)/n':>17} {'ratio vs prev':>14}")

    previous = None
    for n in SIZES:
        variance = float(np.mean([
            bootstrap_mean_variance(pareto_sample(rng, n, a), rng)
            for _ in range(N_REPLICATES)
        ]))
        ratio = "-" if previous is None else f"{variance / previous:.3f}"
        print(f"{n:>7} {variance:21.6f} {theoretical(a, n):17.6f} {ratio:>14}")
        previous = variance


# Takes a minute or two: 40 replicates x 5,000 resamples at each size.
for label, shape in SHAPES.items():
    run(label, shape)
print("\nSample size quadruples each row, so 1/n predicts a ratio near 0.25.")

## Week 3 — Event study and differences-in-differences

### Q1. How would we test for a change in the second derivative as well? (`homework_3.1.csv`)

Add s² and s²·post to the event-study regression, with s = time − 50. The s²·post coefficient is half the change in the second derivative at the event, and an F-test checks whether both curvature terms improve on the slope-break model.

In [ ]:
TIME = "time"
SERIES = ["value1", "value2", "value3"]
EVENT = 50


def curvature_jump(df, series):
    """y = a + b*s + c*s^2 + d*post + e*s*post + f*s^2*post, s = time - 50.
    d, e, 2f are the jumps in value, slope, and second derivative at the event."""
    s = df[TIME] - EVENT
    post = (df[TIME] >= EVENT).astype(float)
    design = sm.add_constant(pd.DataFrame({
        "s": s, "s2": s**2, "post": post, "s_post": s * post, "s2_post": s**2 * post}))
    return sm.OLS(df[series], design).fit()


def compare_nested(df, series):
    """F-test: does adding both curvature terms improve on the slope-break model?"""
    s = df[TIME] - EVENT
    post = (df[TIME] >= EVENT).astype(float)
    small = sm.add_constant(pd.DataFrame({"s": s, "post": post, "s_post": s * post}))
    restricted = sm.OLS(df[series], small).fit()
    return curvature_jump(df, series).compare_f_test(restricted)

In [ ]:
event = pd.read_csv(CSVS / "homework_3.1.csv")
print(f"{'series':<7} {'2f (d2 chg)':>12} {'t':>6} {'p':>7} | {'F':>6} {'p':>7}")
for s in SERIES:
    fit = curvature_jump(event, s)
    f_stat, f_p, _ = compare_nested(event, s)
    print(f"{s:<7} {2 * fit.params['s2_post']:12.5f} {fit.tvalues['s2_post']:6.2f} "
          f"{fit.pvalues['s2_post']:7.3f} | {f_stat:6.2f} {f_p:7.3f}")

### Q2. Create your own scenario that illustrates differences-in-differences

A SaaS company ships a new onboarding checklist to its US accounts in month 7. EU accounts wait on a privacy review, so they keep the old flow and serve as the control. Outcome: support tickets per account per month.

In [ ]:
N_ACCOUNTS = 200          # per region
MONTHS = 12
LAUNCH = 7                # first month with the checklist
TRUE_EFFECT = -0.8        # tickets per account per month


def simulate(seed=SEED):
    """Tickets fall ~0.1/month in both regions as the product matures (parallel
    trends). US accounts start higher; the checklist cuts 0.8 more from launch."""
    rng = np.random.default_rng(seed)
    rows = []
    for us in (0, 1):
        baseline = 4.0 + 1.0 * us
        for account in range(N_ACCOUNTS):
            account_level = rng.normal(0, 1.0)          # some accounts just file more
            for month in range(1, MONTHS + 1):
                post = int(month >= LAUNCH)
                mean = baseline + account_level - 0.1 * month + TRUE_EFFECT * us * post
                rows.append({"account": f"{us}-{account}", "us": us, "month": month,
                             "post": post, "tickets": mean + rng.normal(0, 1.0)})
    return pd.DataFrame(rows)


def diff_in_diff(df):
    """tickets ~ us + post + us*post, SEs clustered by account (12 rows each)."""
    design = sm.add_constant(df[["us", "post"]].assign(us_post=df["us"] * df["post"]))
    return sm.OLS(df["tickets"], design).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(df["account"])[0]})


def pre_trend(df):
    """Before launch only: does the US trend differ from the EU trend?"""
    pre = df[df["post"] == 0]
    design = sm.add_constant(pre[["us", "month"]].assign(us_month=pre["us"] * pre["month"]))
    return sm.OLS(pre["tickets"], design).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(pre["account"])[0]})

In [ ]:
tickets = simulate()
means = tickets.groupby(["us", "post"])["tickets"].mean().unstack()
means.index = ["EU (control)", "US (treated)"]
means.columns = ["before", "after"]
means["change"] = means["after"] - means["before"]
print(means.round(3))

naive = means.loc["US (treated)", "change"]
manual = naive - means.loc["EU (control)", "change"]
fit = diff_in_diff(tickets)
lo, hi = fit.conf_int().loc["us_post"]
trend = pre_trend(tickets)

print(f"\nnaive US before/after:     {naive:.4f}")
print(f"diff-in-diff (by hand):    {manual:.4f}")
print(f"diff-in-diff (regression): {fit.params['us_post']:.4f}  "
      f"se {fit.bse['us_post']:.4f}  t {fit.tvalues['us_post']:.2f}  "
      f"95% CI [{lo:.3f}, {hi:.3f}]  p {fit.pvalues['us_post']:.1e}")
print(f"pre-launch trend gap:      {trend.params['us_month']:.4f}  "
      f"se {trend.bse['us_month']:.4f}  p {trend.pvalues['us_month']:.3f}")

## Week 4 — Instrumental variables and a discontinuity in college admission

### Q1. Divide the range of W into multiple ranges (`homework_4.1.csv`)

Split W into groups, get the instrument's effect in each, and average. Groups that each cover the same range of W (`pd.cut`) leave only a few rows at the edges, which gives wild results, so `pd.qcut` puts the same number of rows in each group.

In [ ]:
def iv_effect(df):
    """(change in Y from Z=0 to Z=1) / (change in X from Z=0 to Z=1)."""
    means = df.groupby("Z")[["X", "Y"]].mean()
    change = means.loc[1] - means.loc[0]
    return change["Y"] / change["X"]


def effects_by_w(df, w_groups):
    """The effect within each W group. Groups with only one Z value are skipped."""
    return np.array([iv_effect(group) for _, group in df.groupby(w_groups, observed=True)
                     if group["Z"].nunique() == 2])


iv = pd.read_csv(CSVS / "homework_4.1.csv")
print(f"effect ignoring W: {iv_effect(iv):.2f}\n")

# pd.cut: every group covers the same range of W. pd.qcut: every group has the same count.
print(f"{'groups':<18} {'average':>8} {'lowest':>8} {'highest':>8}")
for label, split, n in [("same range", pd.cut, 20), ("same range", pd.cut, 40),
                        ("same count", pd.qcut, 20), ("same count", pd.qcut, 100)]:
    effects = effects_by_w(iv, split(iv["W"], n))
    print(f"{f'{label} x{n}':<18} {effects.mean():8.2f} {effects.min():8.2f} {effects.max():8.2f}")

### Q2. Plot college outcome (Y) vs. test score (X) near 80, compared with logistic regression (`homework_4.2.a/b.csv`)

Plot Y vs. X from 75 to 85. Y is 0/1, so the dots are the share admitted in each half-point of score, against the probability predicted by a logistic regression (Y ~ X). The curve is smooth, so it cannot follow the jump at 80.

In [ ]:
CUTOFF = 80
LOW, HIGH = CUTOFF - 5, CUTOFF + 5   # plot 75 to 85
DOT_WIDTH = 0.5                      # each dot covers half a point of score
PLOT = Path("week4_reflection.png")

INK, MUTED, GRID, BLUE = "#0b0b0b", "#52514e", "#e5e4df", "#2a78d6"


def near_cutoff(name):
    df = pd.read_csv(CSVS / name)
    df.columns = ["X", "Y"]
    return df[df["X"].between(LOW, HIGH)]


def share_admitted(df, low, high):
    return df.loc[(df["X"] >= low) & (df["X"] < high), "Y"].mean()


def draw(ax, df, title):
    # Dots: share admitted in each half-point of score.
    edges = np.arange(LOW, HIGH + DOT_WIDTH, DOT_WIDTH)
    dots = df.groupby(pd.cut(df["X"], edges, right=False), observed=True)["Y"].mean()
    ax.scatter(edges[:-1] + DOT_WIDTH / 2, dots * 100, s=22, color=MUTED, label="Actual % admitted")

    # Line: probability predicted by logistic regression.
    fit = smf.logit("Y ~ X", df).fit(disp=0)
    scores = pd.DataFrame({"X": np.linspace(LOW, HIGH, 200)})
    ax.plot(scores["X"], fit.predict(scores) * 100, color=BLUE, lw=2.5,
            label="Logistic regression prediction")

    # Label: actual share within one point on each side of 80 vs. the prediction at 80.
    below = share_admitted(df, CUTOFF - 1, CUTOFF)
    above = share_admitted(df, CUTOFF, CUTOFF + 1)
    at_cutoff = fit.predict(pd.DataFrame({"X": [CUTOFF]})).iloc[0]
    label = (f"Actual: jumps {below:.0%} to {above:.0%} at {CUTOFF}\n"
             f"Logistic at {CUTOFF}: {at_cutoff:.0%}")
    ax.text(LOW + 0.2, 5, label, color=INK, fontsize=9)
    print(f"{title}: {label.replace(chr(10), '   ')}")

    ax.axvline(CUTOFF, color=MUTED, lw=1, ls=":")
    ax.set_title(title, color=INK, loc="left")
    ax.set_xlabel("Test score", color=MUTED)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f}%")
    ax.grid(axis="y", color=GRID, lw=1)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, d in zip(axes, ["a", "b"]):
    draw(ax, near_cutoff(f"homework_4.2.{d}.csv"), f"Dataset {d}")

axes[0].set_ylabel("Chance of getting into college", color=MUTED)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig(PLOT, dpi=150)
plt.show()